In [25]:
# ============================================================
# DLBDSEDA02 - Project: Data Analysis
# Phase 2: Development and Reflection Phase
# Task 1: NLP-Based Analysis of Complaint Texts
# Author: Shantanu Taralekar
#
# Objective:
# This notebook analyzes a collection of complaint-like texts using NLP.
# The workflow follows the Phase 1 concept:
# 1. Load complaint/customer feedback dataset
# 2. Preprocess text data
# 3. Convert text into numerical vectors using BoW and TF-IDF
# 4. Extract topics using LDA and NMF
# 5. Compare and discuss the results
# ============================================================

# Install required libraries

In [50]:
# ============================================================
# Install required Python libraries
# ============================================================

!pip install pandas numpy nltk scikit-learn gensim matplotlib wordcloud --quiet

# Import libraries

In [27]:
# ============================================================
# Import required libraries
# ============================================================

import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt

# NLTK libraries for text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Scikit-learn libraries for vectorization and topic modelling
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF

# Optional: word cloud visualization
from wordcloud import WordCloud

# Download required NLTK resources
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

# Upload dataset

In [28]:
# ============================================================
# Upload the dataset
# ============================================================

from google.colab import drive
import pandas as pd
import os

drive.mount('/content/drive')

file_path = "/content/drive/MyDrive/DLBDSEDA02_Phase2/complaints.csv"

# Check if the file exists before attempting to read it
if os.path.exists(file_path):
    try:
        df = pd.read_csv(file_path)
        print("Dataset loaded successfully!")
        display(df.head())
    except OSError as e:
        print(f"An OSError occurred while reading the file: {e}")
        print("This often indicates a temporary issue with Google Drive connectivity.")
        print("Please ensure your Google Drive connection is stable and try re-running this cell.")
else:
    print(f"Error: The file '{file_path}' does not exist.")
    print("Please verify the file path and ensure the file is in your Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_5088/4253138181.py:16: DtypeWarning: Columns (4,5,6,11,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


Dataset loaded successfully!


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,05/10/2019,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,NaN,NaN,NAVY FEDERAL CREDIT UNION,FL,328XX,Older American,NaN,Web,05/10/2019,In progress,Yes,NaN,3238275
1,05/10/2019,Checking or savings account,Other banking product or service,Managing an account,Deposits and withdrawals,NaN,NaN,BOEING EMPLOYEES CREDIT UNION,WA,98204,NaN,NaN,Referral,05/10/2019,Closed with explanation,Yes,NaN,3238228
2,05/10/2019,Debt collection,Payday loan debt,Communication tactics,Frequent or repeated calls,NaN,NaN,CURO Intermediate Holdings,TX,751XX,NaN,NaN,Web,05/10/2019,Closed with explanation,Yes,NaN,3237964
3,05/10/2019,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Old information reappears or never goes away,NaN,NaN,Ad Astra Recovery Services Inc,LA,708XX,NaN,NaN,Web,05/10/2019,Closed with explanation,Yes,NaN,3238479
4,05/10/2019,Checking or savings account,Checking account,Managing an account,Banking errors,NaN,NaN,ALLY FINANCIAL INC.,AZ,85205,NaN,NaN,Postal mail,05/10/2019,In progress,Yes,NaN,3238460


# Step 4: Inspect dataset columns

In [29]:
# ============================================================
# Step 4: Inspect dataset columns
# ============================================================
# Purpose:
# This step checks the structure of the dataset.
# It helps identify the correct column that contains complaint text.
# For this dataset, the main NLP text column should be:
# "Consumer complaint narrative"
# ============================================================

print("Dataset shape:", df.shape)

print("\nColumn names:")
for column in df.columns:
    print(column)

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (1282355, 18)

Column names:
Date received
Product
Sub-product
Issue
Sub-issue
Consumer complaint narrative
Company public response
Company
State
ZIP code
Tags
Consumer consent provided?
Submitted via
Date sent to company
Company response to consumer
Timely response?
Consumer disputed?
Complaint ID

First 5 rows:


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,05/10/2019,Checking or savings account,Checking account,Managing an account,Problem using a debit or ATM card,NaN,NaN,NAVY FEDERAL CREDIT UNION,FL,328XX,Older American,NaN,Web,05/10/2019,In progress,Yes,NaN,3238275
1,05/10/2019,Checking or savings account,Other banking product or service,Managing an account,Deposits and withdrawals,NaN,NaN,BOEING EMPLOYEES CREDIT UNION,WA,98204,NaN,NaN,Referral,05/10/2019,Closed with explanation,Yes,NaN,3238228
2,05/10/2019,Debt collection,Payday loan debt,Communication tactics,Frequent or repeated calls,NaN,NaN,CURO Intermediate Holdings,TX,751XX,NaN,NaN,Web,05/10/2019,Closed with explanation,Yes,NaN,3237964
3,05/10/2019,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Old information reappears or never goes away,NaN,NaN,Ad Astra Recovery Services Inc,LA,708XX,NaN,NaN,Web,05/10/2019,Closed with explanation,Yes,NaN,3238479
4,05/10/2019,Checking or savings account,Checking account,Managing an account,Banking errors,NaN,NaN,ALLY FINANCIAL INC.,AZ,85205,NaN,NaN,Postal mail,05/10/2019,In progress,Yes,NaN,3238460


# Step 5: Select complaint text column

In [30]:
# ============================================================
# Step 5: Select complaint text column
# ============================================================
# Purpose:
# The column "Consumer complaint narrative" contains the actual
# written complaint text submitted by consumers.
#
# Some rows contain missing values because not every complaint
# includes a public written narrative. These missing rows are removed.
# ============================================================

text_column = "Consumer complaint narrative"

# Select only the complaint narrative column
complaints_df = df[[text_column]].copy()

# Rename the column to a simple project-friendly name
complaints_df = complaints_df.rename(columns={text_column: "complaint_text"})

# Remove rows where complaint text is missing
complaints_df = complaints_df.dropna(subset=["complaint_text"])

# Remove duplicate complaint texts
complaints_df = complaints_df.drop_duplicates(subset=["complaint_text"])

# Reset index
complaints_df = complaints_df.reset_index(drop=True)

print("Dataset shape after selecting complaint texts:", complaints_df.shape)

display(complaints_df.head())

Dataset shape after selecting complaint texts: (366945, 1)


,complaint_text
0,The Summer of XX/XX/2018 I was denied a mortga...
1,There are many mistakes appear in my report wi...
2,There is an account reporting on my credit rep...
3,The reason for my writing is to inform you tha...
4,XXXX and Transunion are reporting incorrectly ...


# Step 6: Take sample for faster processing

In [31]:
# ============================================================
# Step 6: Create a sample dataset for analysis
# ============================================================
# Purpose:
# The full dataset is large. A sample is used to make processing
# faster in Google Colab while still demonstrating the complete
# NLP workflow.
#
# The random_state value makes the sample reproducible.
# ============================================================

sample_size = 3000

if len(complaints_df) > sample_size:
    complaints_df = complaints_df.sample(n=sample_size, random_state=42).reset_index(drop=True)

print("Final dataset size used for analysis:", complaints_df.shape)

display(complaints_df.head())

Final dataset size used for analysis: (3000, 1)


,complaint_text
0,I live in the city of XXXX Maryland. The City ...
1,My account balance on my capital one checking ...
2,multiple attempts to collect on a debt that ha...
3,I have informed Transunion that the Alabama XX...
4,I pulled my credit report on XX/XX/2018 and no...


# Step 7: Define text cleaning function

In [51]:
# ============================================================
# Step 7: Define text preprocessing function
# ============================================================
# Purpose:
# This function cleans raw complaint text and prepares it for NLP.
#
# Cleaning steps:
# 1. Convert text to lowercase
# 2. Remove URLs
# 3. Remove numbers
# 4. Remove punctuation
# 5. Remove special characters
# 6. Remove standard and custom stopwords
# 7. Remove redacted dataset words starting with "xx"
# 8. Apply lemmatization
#
# These steps follow the Phase 1 concept and improve the quality
# of vectorization and topic modelling.
# ============================================================

# Load standard English stopwords from NLTK
stop_words = set(stopwords.words("english"))

# Add dataset-specific custom stopwords
# The Consumer Complaint dataset contains redacted words such as
# "xxxx" and similar forms. These words do not provide useful meaning
# for topic modelling, so they are removed.
custom_stopwords = {
    "xxxx", "xxxxxxxx", "xxxxxxxxxxxx", "xx", "xxx",
    "would", "could", "also", "please",
    "said", "told", "call", "called", "phone",
    "date", "number", "name", "address", "filed"
}

# Combine standard stopwords with custom stopwords
stop_words = stop_words.union(custom_stopwords)

# Create lemmatizer object
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    """
    Clean and preprocess a single complaint text.

    Parameters:
    text (str): Raw complaint text.

    Returns:
    str: Cleaned complaint text.
    """

    # Convert text to string and lowercase
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Keep only alphabetic characters and spaces
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenize text
    words = text.split()

    # Remove standard/custom stopwords, very short words, and redacted words
    words = [
        word for word in words
        if word not in stop_words
        and len(word) > 2
        and not word.startswith("xx")
    ]

    # Lemmatize words
    words = [lemmatizer.lemmatize(word) for word in words]

    # Join words back into a single cleaned sentence
    cleaned_text = " ".join(words)

    return cleaned_text

# Step 8: Apply text preprocessing

In [52]:
# ============================================================
# Step 8: Apply preprocessing to complaint texts
# ============================================================
# Purpose:
# This step creates a new column named "clean_text".
# The cleaned text will be used for Bag of Words, TF-IDF,
# LDA, and NMF.
# ============================================================

complaints_df["clean_text"] = complaints_df["complaint_text"].apply(clean_text)

# Remove rows where cleaned text became empty after preprocessing
complaints_df = complaints_df[complaints_df["clean_text"].str.strip() != ""].reset_index(drop=True)

print("Dataset shape after text preprocessing:", complaints_df.shape)

display(complaints_df[["complaint_text", "clean_text"]].head())

Dataset shape after text preprocessing: (2999, 4)


,complaint_text,clean_text
0,I live in the city of XXXX Maryland. The City ...,live city maryland city offer tax incentive ho...
1,My account balance on my capital one checking ...,account balance capital one checking account o...
2,multiple attempts to collect on a debt that ha...,multiple attempt collect debt charged mistakin...
3,I have informed Transunion that the Alabama XX...,informed transunion alabama account belong dif...
4,I pulled my credit report on XX/XX/2018 and no...,pulled credit report noticed collection accoun...


# Step 9: Basic text statistics

In [34]:
# ============================================================
# Step 9: Basic text statistics
# ============================================================
# Purpose:
# This step gives a simple overview of the cleaned complaint texts.
# It helps show that the dataset is suitable for topic analysis.
# ============================================================

complaints_df["word_count"] = complaints_df["clean_text"].apply(lambda text: len(text.split()))

print("Average words per cleaned complaint:", round(complaints_df["word_count"].mean(), 2))
print("Minimum words:", complaints_df["word_count"].min())
print("Maximum words:", complaints_df["word_count"].max())

display(complaints_df[["clean_text", "word_count"]].head())

Average words per cleaned complaint: 97.88
Minimum words: 2
Maximum words: 1421


,clean_text,word_count
0,live city xxxx maryland city offer tax incenti...,302
1,account balance capital one checking account o...,11
2,multiple attempt collect debt charged mistakin...,43
3,informed transunion alabama xxxx xxxx account ...,25
4,pulled credit report xxxx noticed collection a...,52


# Step 10: Bag of Words vectorization

In [53]:
# ============================================================
# Step 10: Vectorization Method 1 - Bag of Words
# ============================================================
# Purpose:
# Bag of Words converts text into a numerical document-term matrix.
# It counts how often each word appears in each complaint.
#
# This vector representation is used as input for LDA.
# ============================================================

bow_vectorizer = CountVectorizer(
    max_df=0.90,
    min_df=5,
    max_features=1000
)

bow_matrix = bow_vectorizer.fit_transform(complaints_df["clean_text"])

print("Bag of Words matrix shape:", bow_matrix.shape)

Bag of Words matrix shape: (2999, 1000)


# Step 11: TF-IDF vectorization

In [54]:
# ============================================================
# Step 11: Vectorization Method 2 - TF-IDF
# ============================================================
# Purpose:
# TF-IDF converts text into numerical values based on word importance.
# It reduces the weight of very common words and gives more importance
# to words that are more specific to certain complaints.
#
# This vector representation is used as input for NMF.
# ============================================================

tfidf_vectorizer = TfidfVectorizer(
    max_df=0.90,
    min_df=5,
    max_features=1000
)

tfidf_matrix = tfidf_vectorizer.fit_transform(complaints_df["clean_text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (2999, 1000)


# Step 12: Helper function for displaying topics

In [37]:
# ============================================================
# Step 12: Helper function to display extracted topics
# ============================================================
# Purpose:
# This function extracts the most important words from each topic.
# It is used for both LDA and NMF topic modelling results.
# ============================================================

def display_topics(model, feature_names, number_of_words=10):
    """
    Display the top words for each extracted topic.

    Parameters:
    model: Trained topic modelling model.
    feature_names: Vocabulary words from vectorizer.
    number_of_words (int): Number of top words per topic.

    Returns:
    pandas.DataFrame: Table containing topic numbers and top words.
    """

    topic_results = []

    for topic_index, topic in enumerate(model.components_):
        top_word_indexes = topic.argsort()[:-number_of_words - 1:-1]
        top_words = [feature_names[i] for i in top_word_indexes]

        topic_results.append({
            "Topic": f"Topic {topic_index + 1}",
            "Top Words": ", ".join(top_words)
        })

        print(f"Topic {topic_index + 1}:")
        print(", ".join(top_words))
        print("-" * 80)

    return pd.DataFrame(topic_results)

# Step 13: LDA topic modelling

In [55]:
# ============================================================
# Step 13: Topic Modelling Technique 1 - LDA
# ============================================================
# Purpose:
# Latent Dirichlet Allocation is used to extract common topics
# from the complaint texts.
#
# LDA works well with Bag of Words because it uses word counts
# to identify probabilistic topic groups.
# ============================================================

number_of_topics = 5

lda_model = LatentDirichletAllocation(
    n_components=number_of_topics,
    random_state=42,
    learning_method="batch",
    max_iter=20
)

lda_model.fit(bow_matrix)

bow_feature_names = bow_vectorizer.get_feature_names_out()

print("LDA Topic Modelling Results")
print("=" * 80)

lda_topics_df = display_topics(
    model=lda_model,
    feature_names=bow_feature_names,
    number_of_words=10
)

display(lda_topics_df)

LDA Topic Modelling Results
Topic 1:
credit, report, account, debt, information, reporting, collection, letter, company, bureau
--------------------------------------------------------------------------------
Topic 2:
payment, loan, pay, late, month, paid, due, balance, time, interest
--------------------------------------------------------------------------------
Topic 3:
mortgage, loan, home, modification, document, property, tax, insurance, time, year
--------------------------------------------------------------------------------
Topic 4:
card, account, credit, charge, fraud, service, bank, one, customer, information
--------------------------------------------------------------------------------
Topic 5:
bank, account, money, check, back, time, well, day, get, asked
--------------------------------------------------------------------------------


,Topic,Top Words
0,Topic 1,"credit, report, account, debt, information, re..."
1,Topic 2,"payment, loan, pay, late, month, paid, due, ba..."
2,Topic 3,"mortgage, loan, home, modification, document, ..."
3,Topic 4,"card, account, credit, charge, fraud, service,..."
4,Topic 5,"bank, account, money, check, back, time, well,..."


# Step 14: NMF topic modelling

In [56]:
# ============================================================
# Step 14: Topic Modelling Technique 2 - NMF
# ============================================================
# Purpose:
# Non-negative Matrix Factorization is used as the second topic
# extraction technique.
#
# NMF works well with TF-IDF because it finds groups of important
# words that commonly appear together.
# ============================================================

nmf_model = NMF(
    n_components=number_of_topics,
    random_state=42,
    init="nndsvda",
    max_iter=500
)

nmf_model.fit(tfidf_matrix)

tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

print("NMF Topic Modelling Results")
print("=" * 80)

nmf_topics_df = display_topics(
    model=nmf_model,
    feature_names=tfidf_feature_names,
    number_of_words=10
)

display(nmf_topics_df)

NMF Topic Modelling Results
Topic 1:
account, card, bank, fee, charge, check, money, closed, chase, day
--------------------------------------------------------------------------------
Topic 2:
loan, mortgage, home, modification, year, interest, student, time, rate, get
--------------------------------------------------------------------------------
Topic 3:
debt, collection, company, letter, collect, validation, received, agency, owe, sent
--------------------------------------------------------------------------------
Topic 4:
credit, report, account, reporting, information, inquiry, equifax, bureau, removed, dispute
--------------------------------------------------------------------------------
Topic 5:
payment, late, month, made, due, statement, fee, paid, pay, balance
--------------------------------------------------------------------------------


,Topic,Top Words
0,Topic 1,"account, card, bank, fee, charge, check, money..."
1,Topic 2,"loan, mortgage, home, modification, year, inte..."
2,Topic 3,"debt, collection, company, letter, collect, va..."
3,Topic 4,"credit, report, account, reporting, informatio..."
4,Topic 5,"payment, late, month, made, due, statement, fe..."


# Step 15: Assign dominant topic to each complaint

In [40]:
# ============================================================
# Step 15: Assign dominant NMF topic to each complaint
# ============================================================
# Purpose:
# This step identifies the strongest NMF topic for each complaint.
# It helps connect the extracted topics back to real complaint texts.
# ============================================================

nmf_topic_values = nmf_model.transform(tfidf_matrix)

complaints_df["dominant_nmf_topic"] = nmf_topic_values.argmax(axis=1) + 1

display(complaints_df[["complaint_text", "clean_text", "dominant_nmf_topic"]].head())

,complaint_text,clean_text,dominant_nmf_topic
0,I live in the city of XXXX Maryland. The City ...,live city xxxx maryland city offer tax incenti...,5
1,My account balance on my capital one checking ...,account balance capital one checking account o...,2
2,multiple attempts to collect on a debt that ha...,multiple attempt collect debt charged mistakin...,4
3,I have informed Transunion that the Alabama XX...,informed transunion alabama xxxx xxxx account ...,3
4,I pulled my credit report on XX/XX/2018 and no...,pulled credit report xxxx noticed collection a...,4


# Step 16: Show example complaints from each topic

In [41]:
# ============================================================
# Step 16: Show example complaints for each topic
# ============================================================
# Purpose:
# This step helps interpret the topic results by showing real
# complaint examples under each topic.
# ============================================================

for topic_number in range(1, number_of_topics + 1):
    print("=" * 100)
    print(f"Example complaints for NMF Topic {topic_number}")
    print("=" * 100)

    examples = complaints_df[
        complaints_df["dominant_nmf_topic"] == topic_number
    ]["complaint_text"].head(2)

    for index, complaint in enumerate(examples, start=1):
        print(f"\nExample {index}:")
        print(complaint[:700])
        print()

Example complaints for NMF Topic 1

Example 1:
I sent XXXX a letter of dispute regarding XXXX public records XXXX XXXX XXXX XXXX. Transunion and XXXX both reported that this information was furnished to them by XXXX, which is a third party vendor. I advised Transunion and XXXX that this is a direct violation of the Fair Credit Reporting Act. I sent a certified letter of dispute to XXXX on XXXX and they signed for the letter on XXXX. I have the actual letter attached to this as proof with no contact from XXXX and it has been more than 30 days. The Tax Liens are {$990.00} and {$1000.00}. I also made contact with Oklahoma XXXX XXXX and they verified that XXXX was never given this information by them. As a result of this, XXXX reported this 


Example 2:
Re : XXXX XXXX XXXX ( AND ) XXXX XXXX Additional information to add to original complaint. BOTH TU and XXXX XXXX are reporting derogatory illegal inaccurate reporting under FCRA. They are both showing ( 2 ) Federal Bankruptcy filing as opp

# Step 17: Compare vectorization methods

In [42]:
# ============================================================
# Step 17: Compare Bag of Words and TF-IDF
# ============================================================
# Purpose:
# This comparison explains why two vectorization methods were used.
# ============================================================

vectorization_comparison = pd.DataFrame({
    "Vectorization Method": ["Bag of Words", "TF-IDF"],
    "Description": [
        "Counts how often each word appears in each complaint.",
        "Measures word importance by considering frequency in one complaint and rarity across all complaints."
    ],
    "Strength": [
        "Simple and suitable for frequency-based topic modelling.",
        "Better at reducing the influence of overly common words."
    ],
    "Used With": ["LDA", "NMF"]
})

display(vectorization_comparison)

,Vectorization Method,Description,Strength,Used With
0,Bag of Words,Counts how often each word appears in each com...,Simple and suitable for frequency-based topic ...,LDA
1,TF-IDF,Measures word importance by considering freque...,Better at reducing the influence of overly com...,NMF


# Step 18: Compare topic modelling methods

In [43]:
# ============================================================
# Step 18: Compare LDA and NMF
# ============================================================
# Purpose:
# This comparison explains the two topic extraction techniques.
# ============================================================

model_comparison = pd.DataFrame({
    "Topic Modelling Method": ["LDA", "NMF"],
    "Input Matrix": ["Bag of Words", "TF-IDF"],
    "Main Idea": [
        "Probabilistic model that represents documents as mixtures of topics.",
        "Matrix factorization method that finds groups of important words."
    ],
    "Result Interpretation": [
        "Produces broader probabilistic topic groups.",
        "Often produces cleaner and more interpretable topic words."
    ]
})

display(model_comparison)

,Topic Modelling Method,Input Matrix,Main Idea,Result Interpretation
0,LDA,Bag of Words,Probabilistic model that represents documents ...,Produces broader probabilistic topic groups.
1,NMF,TF-IDF,Matrix factorization method that finds groups ...,Often produces cleaner and more interpretable ...


# Step 19: Save result files

In [57]:
# ============================================================
# Step 19: Save result files
# ============================================================
# Purpose:
# These CSV files provide evidence of implementation results.
# They can be uploaded to GitHub with the notebook.
# ============================================================

lda_topics_df.to_csv("lda_topics.csv", index=False)
nmf_topics_df.to_csv("nmf_topics.csv", index=False)
vectorization_comparison.to_csv("vectorization_comparison.csv", index=False)
model_comparison.to_csv("model_comparison.csv", index=False)

print("Result files saved successfully:")
print("1. lda_topics.csv")
print("2. nmf_topics.csv")
print("3. vectorization_comparison.csv")
print("4. model_comparison.csv")

Result files saved successfully:
1. lda_topics.csv
2. nmf_topics.csv
3. vectorization_comparison.csv
4. model_comparison.csv


# Step 20: Create requirements.txt

In [45]:
# ============================================================
# Step 20: Create requirements.txt
# ============================================================
# Purpose:
# This file lists the required Python libraries.
# It improves reproducibility when the project is shared on GitHub.
# ============================================================

requirements_text = """
pandas
numpy
nltk
scikit-learn
gensim
matplotlib
wordcloud
"""

with open("requirements.txt", "w") as file:
    file.write(requirements_text.strip())

print("requirements.txt created successfully.")

requirements.txt created successfully.


# Step 21: Create README.md

In [46]:
# ============================================================
# Step 21: Create README.md
# ============================================================
# Purpose:
# This file documents the GitHub repository professionally.
# It explains the project objective, dataset, methodology,
# technologies, outputs, and how to run the code.
# ============================================================

readme_text = """
# DLBDSEDA02 - NLP-Based Complaint Topic Analysis

## Project Overview

This project was developed for the IU course **Project: Data Analysis (DLBDSEDA02)**.
The selected portfolio task is **Task 1: Use NLP techniques to analyze a collection of texts**.

The objective of this project is to analyze complaint texts using Natural Language Processing techniques. The project identifies recurring topics from unstructured complaint narratives and presents them in a structured format.

## Dataset

The project uses the **Consumer Complaint Database** dataset from Kaggle.
The dataset contains complaint records related to financial products and services.
For this analysis, the column **Consumer complaint narrative** is used because it contains the written complaint text.

## Methodology

The analysis follows these steps:

1. Load the complaint dataset.
2. Select the complaint narrative column.
3. Remove missing and duplicate complaint texts.
4. Clean and preprocess the text.
5. Convert text into numerical vectors using:
   - Bag of Words
   - TF-IDF
6. Extract prevalent topics using:
   - Latent Dirichlet Allocation
   - Non-negative Matrix Factorization
7. Compare the results.
8. Save topic modelling outputs as CSV files.

## Text Preprocessing

The preprocessing step includes:

- Lowercase conversion
- URL removal
- Number removal
- Punctuation removal
- Special character removal
- Stopword removal
- Lemmatization

## Technologies Used

- Python
- Google Colab
- pandas
- NumPy
- NLTK
- scikit-learn
- gensim
- matplotlib
- wordcloud

## Output Files

The notebook generates the following output files:

- `lda_topics.csv`
- `nmf_topics.csv`
- `vectorization_comparison.csv`
- `model_comparison.csv`
- `requirements.txt`

## How to Run

1. Open `nlp_complaint_topic_analysis.ipynb` in Google Colab.
2. Mount Google Drive.
3. Place `complaints.csv` inside the selected Google Drive folder.
4. Run all notebook cells from top to bottom.
5. Review the generated topic modelling results.

## Author

Shantanu Taralekar
"""

with open("README.md", "w") as file:
    file.write(readme_text.strip())

print("README.md created successfully.")

README.md created successfully.


# Step 22: Download output files

In [58]:
# ============================================================
# Step 22: Download output files
# ============================================================
# Purpose:
# These files should be downloaded and uploaded to GitHub.
# ============================================================

from google.colab import files

files.download("lda_topics.csv")
files.download("nmf_topics.csv")
files.download("vectorization_comparison.csv")
files.download("model_comparison.csv")
files.download("requirements.txt")
files.download("README.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>